# =====================================================
# 1. Configuración inicial
# =====================================================


In [1]:
# =====================================================
# 1. Importación de librerías
# =====================================================

import cv2
import mediapipe as mp
import numpy as np



## 2. Inicialización de MediaPipe Hands y OpenCV

Se inicializan los módulos de **MediaPipe Hands** para la detección y rastreo de manos, y **OpenCV** para la captura de video en tiempo real.


In [2]:
# =====================================================
# 2. Configuración de MediaPipe
# =====================================================

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


## 3. Función auxiliar: conteo de dedos

La siguiente función identifica cuántos dedos están extendidos, basándose en la posición relativa de las articulaciones.


In [3]:
# =====================================================
# 3. Función de conteo de dedos
# =====================================================

def contar_dedos(hand_landmarks):
    dedos = [0, 0, 0, 0, 0]  # pulgar → meñique
    landmarks = hand_landmarks.landmark

    # Pulgar: comparar eje x (mano derecha vs izquierda)
    if landmarks[4].x < landmarks[3].x:
        dedos[0] = 1  # Pulgar extendido

    # Otros dedos: comparar eje y de la punta con la base
    for i, tip in enumerate([8, 12, 16, 20]):  # índice, medio, anular, meñique
        if landmarks[tip].y < landmarks[tip - 2].y:
            dedos[i + 1] = 1

    return sum(dedos)


## 4. Bucle principal de detección

Captura video desde la cámara, procesa cada fotograma con MediaPipe y muestra los resultados visuales en pantalla.


In [ ]:
# =====================================================
# 4. Detección en tiempo real
# =====================================================

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            dedos = contar_dedos(hand_landmarks)

            # Mostrar conteo en pantalla
            cv2.putText(frame, f'Dedos: {dedos}', (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

            # Ejemplo: cambiar color de fondo si hay 5 dedos extendidos
            if dedos == 5:
                cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 255, 0), 50)
            elif dedos == 0:
                cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 50)

    cv2.imshow('Detección de gestos - Taller 7', frame)
    if cv2.waitKey(1) & 0xFF == 27:  # Tecla ESC para salir
        break

cap.release()
cv2.destroyAllWindows()


## 5. Extensión: detección de gestos simples

Se pueden agregar condiciones para detectar gestos específicos, por ejemplo:
- ✊ puño cerrado (0 dedos)
- ✋ mano abierta (5 dedos)
- 👉 apuntar (1 dedo)
- 🤘 gesto personalizado (índice + meñique)

Estos gestos pueden mapearse a acciones visuales, como mover objetos, cambiar colores o activar eventos en una interfaz.


In [4]:
# =====================================================
# 7. Gestos con cámara web (MediaPipe Hands)
# =====================================================
# Detección de manos en tiempo real (mediapipe + opencv).
# Conteo de dedos, distancias y gestos.
# Mapeo gesto → acción visual.
# Minijuego o interfaz gestual sin hardware adicional.
# =====================================================

import cv2
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.5
)

# =====================================================
def contar_dedos(hand_landmarks):
    dedos = 0
    tips = [4, 8, 12, 16, 20]
    lm = hand_landmarks.landmark

    # Pulgar
    if lm[tips[0]].x < lm[tips[0] - 1].x:
        dedos += 1

    # Otros dedos
    for id in range(1, 5):
        if lm[tips[id]].y < lm[tips[id] - 2].y:
            dedos += 1

    return dedos


# =====================================================
# Ejemplo de mapeo gesto → acción
# =====================================================
def reconocer_gesto(n_dedos):
    if n_dedos == 0:
        return "PUÑO"
    elif n_dedos == 1:
        return "SEÑALAR"
    elif n_dedos == 2:
        return "PAZ"
    elif n_dedos == 5:
        return "ABIERTO"
    else:
        return "INDEFINIDO"


# =====================================================
# Detección en tiempo real
# =====================================================
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            dedos = contar_dedos(hand_landmarks)
            gesto = reconocer_gesto(dedos)

            # Mostrar conteo y gesto
            cv2.putText(frame, f'Dedos: {dedos} | Gesto: {gesto}', (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)

            # Reacción visual según el gesto
            if gesto == "ABIERTO":
                cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 255, 0), 50)
            elif gesto == "PUÑO":
                cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 50)
            elif gesto == "PAZ":
                cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (255, 0, 255), 50)
            elif gesto == "SEÑALAR":
                cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 255, 255), 50)

    else:
        cv2.putText(frame, "No se detectan manos", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 100, 255), 2)

    cv2.imshow('Detección de gestos - Taller 7', frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()



KeyboardInterrupt: 

## 6. Conclusiones técnicas

La combinación de **MediaPipe Hands** y **OpenCV** permite implementar una interfaz gestual funcional sin hardware especializado.  
El sistema detecta movimientos en tiempo real, con buena precisión y rendimiento, apto para integrarse en minijuegos o visualizaciones interactivas.

Posibles mejoras:
- Suavizado temporal de detecciones.
- Reconocimiento de secuencias gestuales.
- Integración con control de cámara en Unity o Processing vía OSC.
